# bench118 — Laya fine-tuning (Kaggle)

Fine-tunes **`convaiinnovations/laya-multilingual`** on the 118.studio tool-calling benchmark (Turkish), then runs the
benchmark on the **held-out test split** for the base and the fine-tuned model side by side.

* Train: 445 tests / 703 steps / 9,630 labelled questions — Test: 192 tests / 314 steps (never seen in training;
  a noisy_ascii copy always stays on the same side as its original).
* Training data comes from `python -m bench118 export-laya` (`data/laya_ft/`).

### Kaggle settings (right sidebar → *Notebook options*)
* **Accelerator:** `GPU T4 x2` (one GPU is used) or `GPU P100`
* **Internet:** `On` (downloads the base checkpoint and `laya`)
* **Input:** add the uploaded `bench118_kaggle.zip` as a dataset (*Add Input → Upload*). Without it the notebook clones
  `https://github.com/ceren15/Bench`.

Outputs in `/kaggle/working`: `laya_bench118_ft.zip` (fine-tuned model), `results_laya.zip` (benchmark report).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 or P100"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q "laya==0.4.0" pyyaml psutil
import laya; print("laya", laya.__version__)

## 1. Project files

In [ ]:
import os, glob, shutil, subprocess
PROJECT = "/kaggle/working/bench118"
hits = glob.glob("/kaggle/input/**/bench118/compiler.py", recursive=True)
if os.path.exists(PROJECT):
    print("using existing", PROJECT)
elif hits:
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src, PROJECT)
    print("copied uploaded project from", src)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ceren15/Bench", PROJECT], check=True)
    print("cloned from GitHub")
os.chdir(PROJECT)
for f in ("data/laya_ft/train.jsonl", "data/laya_ft/eval.jsonl", "data/laya_ft/dataset_test.jsonl", "data/tools.json"):
    assert os.path.exists(f), f"missing {f} — upload bench118_kaggle.zip or run `python -m bench118 export-laya` first"
!wc -l data/laya_ft/*.jsonl

## 2. Base checkpoint

In [ ]:
from huggingface_hub import snapshot_download
BASE_DIR = snapshot_download("convaiinnovations/laya-multilingual")
print(BASE_DIR); print(open(os.path.join(BASE_DIR, "rl_agent_config.json")).read()[:400])

## 3. Fine-tune

Token budgets were measured on this data with the checkpoint's tokenizer: the routing question with 11 tool options is
~593 tokens, the longest state 510. `head_max_len=640` keeps every option description intact (the default 256 would cut
each to ~21 tokens) and `max_len=1152` leaves room for the whole state. `shuffle_options=("choice",)` draws a new option
order per epoch — the base model was strongly position-biased in our zero-shot run.

In [ ]:
import json, time
import laya.train as lt
from laya.train import TrainConfig, finetune

# laya 0.4.0 bug: finetune() scores the base model on eval/calibration items while the model
# is still on CPU (it is moved to the GPU only inside train_model) -> load it on the GPU.
_load_checkpoint = lt.load_checkpoint
def _load_checkpoint_on_gpu(model_dir):
    model, tok, cfg = _load_checkpoint(model_dir)
    return model.to("cuda"), tok, cfg
lt.load_checkpoint = _load_checkpoint_on_gpu

MAX_LEN, HEAD_MAX_LEN = 1152, 640
FT_DIR = "/kaggle/working/laya_bench118_ft"
cfg = TrainConfig(epochs=4, micro_batch=8, grad_accum=8, max_len=MAX_LEN, head_max_len=HEAD_MAX_LEN,
                  shuffle_options=("choice",), eval_data="data/laya_ft/eval.jsonl", seed=118, log_every=50)
t0 = time.time()
summary = finetune("data/laya_ft/train.jsonl", BASE_DIR, FT_DIR, cfg, device="cuda")
print(f"done in {(time.time() - t0) / 60:.1f} min")
print(json.dumps(summary, indent=2, default=str)[:4000])

## 4. Benchmark on the held-out test split: base vs fine-tuned

Same compiled questions, same token budgets, 3 repeats with option-order shuffling (measures order sensitivity).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, PROJECT)
from bench118 import report
from bench118.adapters import build
from bench118.metrics import dumps
from bench118.runner import RunConfig, Runner
from bench118.schema import Catalog, TestCase

cat = Catalog()
TEST = "data/laya_ft/dataset_test.jsonl"
tests = TestCase.load_all(Path(TEST))
rc = RunConfig(repeats=3, warmup=5, perturb_option_order=True, question_language="tr", dataset=TEST)
OUT = Path("/kaggle/working/results/laya_base_vs_ft")
runner = Runner(cat, tests, rc, OUT)
common = {"adapter": "laya_local", "subfolder": None, "device": "cuda", "max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN}
adapters = [build({"name": "laya-ml-base", "repo": BASE_DIR, **common}),
            build({"name": "laya-ml-ft", "repo": FT_DIR, **common})]
meta = runner.meta(adapters)
(OUT / "run_meta.json").write_text(dumps(meta), encoding="utf-8")
results = []
for a in adapters:
    print(f"→ {a.name}: {len(tests)} tests …", flush=True)
    res = await runner.run_model(a)
    s, m = res["score"], res["metrics"]
    print(f"  overall={s['overall']:.1f} pass={m['test_pass_rate']:.3f} routing={m['routing_accuracy']:.3f} "
          f"tools={m['tool_selection_acc']:.3f} errors={m['errors']}")
    results.append(res)
md_path = report.write(results, meta, OUT)
print(open(md_path).read())

## 5. Download

In [ ]:
shutil.make_archive("/kaggle/working/laya_bench118_ft", "zip", FT_DIR)
shutil.make_archive("/kaggle/working/results_laya", "zip", "/kaggle/working/results")
!ls -la /kaggle/working/*.zip